# RF Monte Carlo CPU Benchmark
独立运行，不导入或修改 Rank-DML。请将整个文件夹放到主机，选择 Python 3.10+ kernel，再按顺序执行。

默认仅跑一组 **12 workers / 40 reps / 500 trees**。环境检查会安装缺失包并保存版本快照。正式执行由下面的运行单元格开始。

In [ ]:
from pathlib import Path
import sys, subprocess, json, csv, os

# 一般 notebook 工作目录即本文件所在目录；若找不到请填写上传后的绝对路径。
BENCH_DIR = Path.cwd()
if not (BENCH_DIR / "benchmark.py").is_file():
    candidates = [BENCH_DIR / "rf_mc_benchmark", Path.home() / "Desktop" / "rf_mc_benchmark"]
    BENCH_DIR = next((p for p in candidates if (p / "benchmark.py").is_file()), BENCH_DIR)
assert (BENCH_DIR / "benchmark.py").is_file(), "请将 BENCH_DIR 改为主机上 rf_mc_benchmark 的绝对路径"
print("Interpreter:", sys.executable)
print("Benchmark:", BENCH_DIR)

def run_live(command):
    """Stream child tqdm output into notebook and stop descendants on interrupt."""
    print("Command:", " ".join(map(str, command)), flush=True)
    process = subprocess.Popen(list(map(str, command)), cwd=BENCH_DIR,
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, encoding="utf-8", errors="replace",
                               env=dict(os.environ, PYTHONIOENCODING="utf-8", PYTHONUNBUFFERED="1"))
    try:
        for line in process.stdout:
            print(line.rstrip(), flush=True)
        result = process.wait()
        if result:
            raise RuntimeError(f"命令退出码 {result}；请检查上方错误与已保存结果")
    except KeyboardInterrupt:
        import psutil
        try:
            parent = psutil.Process(process.pid)
            children = parent.children(recursive=True)
            for child in children:
                try:
                    child.terminate()
                except psutil.NoSuchProcess:
                    pass
            parent.terminate()
            _, alive = psutil.wait_procs(children + [parent], timeout=5)
            for child in alive:
                try:
                    child.kill()
                except psutil.NoSuchProcess:
                    pass
        except psutil.NoSuchProcess:
            pass
        print("已停止本次子进程。确认退出后删除对应 .lock，重新运行可续跑；正式计时请用新 RUN_ID 或 FORCE=True。")
        raise

run_live([sys.executable, "setup_env.py"])
print("环境检查通过，可以保存云端环境/镜像。版本快照在 environment_snapshot/。")

## 选择配置
`MODE='single'` 默认只跑 WORKERS；`MODE='suite'` 顺序跑 12/16/24/32 和 n=2000,w=24；`MODE='smoke'` 仅跑极小检查。

同批次必须保持 RUN_ID/SEED 一致。重复相同命令会读取断点；需要重新计时改 RUN_ID 或设置 FORCE=True。不要同时在其他终端运行这组配置。

In [ ]:
MODE = "single"  # "single" / "suite" / "smoke"
N, REPS, TREES, WORKERS = 1000, 40, 500, 12
SEED = 20261117
RUN_ID = "cpu_compare"
FORCE = False
OUTPUT = BENCH_DIR / "results"

if MODE == "single":
    CONFIGS = [(N, REPS, WORKERS, TREES)]
elif MODE == "suite":
    CONFIGS = [(1000, 40, w, 500) for w in (12, 16, 24, 32)] + [(2000, 40, 24, 500)]
elif MODE == "smoke":
    CONFIGS = [(200, 2, 2, 10)]
    RUN_ID = "smoke"
else:
    raise ValueError("未知 MODE")
print("顺序执行:", CONFIGS)
print("输出:", OUTPUT / RUN_ID)

In [ ]:
# 正式开始；本单元格会持续输出 tqdm 进度。不要重复点击运行。
for n, reps, workers, trees in CONFIGS:
    command = [sys.executable, "benchmark.py", "--n", n, "--reps", reps,
               "--workers", workers, "--trees", trees, "--seed", SEED,
               "--run-id", RUN_ID, "--output", OUTPUT]
    if FORCE:
        command.append("--force")
    run_live(command)
print("全部选定配置完成。")

## 查看结果与矢量图
worker 加速比仅匹配本机同批次的 n/reps/trees/seed 与环境。n=2000 若没有对应 workers=12 baseline，加速比留空。

In [ ]:
from IPython.display import display, Markdown, FileLink
for n, reps, workers, trees in CONFIGS:
    folder = OUTPUT / RUN_ID / f"n{n}_r{reps}_t{trees}_w{workers}_s{SEED}"
    if (folder / "benchmark_report.md").exists():
        display(Markdown((folder / "benchmark_report.md").read_text(encoding="utf-8")))
        for name in ("replications.csv", "benchmark_summary.csv", "replication_durations.pdf"):
            display(FileLink(str(folder / name)))
        # Notebook 内直接预览耗时；正式交付仍为主程序生成的矢量 PDF。
        import matplotlib.pyplot as plt
        with (folder / "replications.csv").open(encoding="utf-8-sig", newline="") as stream:
            rows = list(csv.DictReader(stream))
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.bar([int(r["replication"]) for r in rows],
               [float(r["elapsed_seconds"]) for r in rows],
               color=["#2274A5" if r["status"] == "success" else "#C44536" for r in rows])
        ax.set(xlabel="Replication", ylabel="Task wall time (seconds)", title=f"n={n}, workers={workers}")
        plt.show()